# Statistiques des Datasets — BCS Determination

Analyse exploratoire des datasets utilisés dans le pipeline BCS :

| Dataset | Usage | Images | Classes |
|---|---|---|---|
| **Stanford Dogs** | Classification de race | ~20 580 | 120 races |
| **Oxford-IIIT Pet** | Segmentation sémantique | ~7 390 | 37 races (chats + chiens) |
| **Reddit Examples** | Test hors distribution | 2 | — |

**Contenu du notebook :**
1. Stanford Dogs — distribution des classes, tailles d'images, réflectances RGB
2. Oxford-IIIT Pet — distribution espèces/races, masques trimap, réflectances
3. Comparaison inter-datasets

In [ ]:
import os
import glob
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from scipy import stats as sp_stats

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"

REPO_ROOT = Path("..").resolve()
STANFORD_DIR = REPO_ROOT / "data" / "Stanford_dogs" / "Images"
OXFORD_IMG_DIR = REPO_ROOT / "data" / "Oxford-IIIT_pet_dataset" / "images"
OXFORD_ANN_DIR = REPO_ROOT / "data" / "Oxford-IIIT_pet_dataset" / "annotations"
OXFORD_TRIMAP_DIR = OXFORD_ANN_DIR / "trimaps"
REDDIT_DIR = REPO_ROOT / "data" / "Reddit_example"

print(f"Stanford Dogs dir : {STANFORD_DIR}  (exists={STANFORD_DIR.exists()})")
print(f"Oxford Pet dir    : {OXFORD_IMG_DIR}  (exists={OXFORD_IMG_DIR.exists()})")
print(f"Reddit dir        : {REDDIT_DIR}  (exists={REDDIT_DIR.exists()})")

---
## 1. Stanford Dogs Dataset (120 races)

### 1.1 Chargement et structure

In [ ]:
# Charger la structure du dataset Stanford Dogs
stanford_breeds = sorted([d.name for d in STANFORD_DIR.iterdir() if d.is_dir()])
stanford_breed_names = [b.split("-", 1)[1].replace("_", " ") for b in stanford_breeds]

stanford_data = []
for breed_dir in sorted(STANFORD_DIR.iterdir()):
    if not breed_dir.is_dir():
        continue
    breed_name = breed_dir.name.split("-", 1)[1].replace("_", " ")
    images = list(breed_dir.glob("*.jpg")) + list(breed_dir.glob("*.JPEG")) + list(breed_dir.glob("*.png"))
    stanford_data.append({
        "breed_folder": breed_dir.name,
        "breed": breed_name,
        "count": len(images),
        "image_paths": images,
    })

df_stanford = pd.DataFrame(stanford_data)
total_stanford = df_stanford["count"].sum()

print(f"Nombre de races  : {len(df_stanford)}")
print(f"Total images     : {total_stanford}")
print(f"Images par race  : min={df_stanford['count'].min()}, max={df_stanford['count'].max()}, "
      f"mean={df_stanford['count'].mean():.1f}, median={df_stanford['count'].median():.0f}, "
      f"std={df_stanford['count'].std():.1f}")
print(f"Ratio desequilibre (max/min) : {df_stanford['count'].max() / df_stanford['count'].min():.2f}")
df_stanford[["breed", "count"]].describe()

### 1.2 Distribution des classes (races)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Bar chart trié par nombre d'images
df_sorted = df_stanford.sort_values("count", ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(df_sorted)))

axes[0].barh(range(len(df_sorted)), df_sorted["count"], color=colors, edgecolor="none")
axes[0].set_yticks(range(0, len(df_sorted), 5))
axes[0].set_yticklabels(df_sorted["breed"].iloc[::5], fontsize=6)
axes[0].set_xlabel("Nombre d'images")
axes[0].set_title("Stanford Dogs — Images par race (120 races)", fontweight="bold")
axes[0].axvline(df_stanford["count"].mean(), color="red", ls="--", lw=1.5, label=f"Moyenne = {df_stanford['count'].mean():.0f}")
axes[0].axvline(df_stanford["count"].median(), color="orange", ls=":", lw=1.5, label=f"Mediane = {df_stanford['count'].median():.0f}")
axes[0].legend(fontsize=9)

# Histogramme de la distribution
axes[1].hist(df_stanford["count"], bins=30, color="steelblue", edgecolor="white", alpha=0.85)
axes[1].axvline(df_stanford["count"].mean(), color="red", ls="--", lw=2, label=f"Moyenne = {df_stanford['count'].mean():.1f}")
axes[1].axvline(df_stanford["count"].median(), color="orange", ls=":", lw=2, label=f"Mediane = {df_stanford['count'].median():.0f}")
axes[1].set_xlabel("Nombre d'images par race")
axes[1].set_ylabel("Nombre de races")
axes[1].set_title("Distribution du nombre d'images par race", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

# Top 10 et Bottom 10
print("=== Top 10 races (les plus representees) ===")
print(df_stanford.nlargest(10, "count")[["breed", "count"]].to_string(index=False))
print(f"\n=== Bottom 10 races (les moins representees) ===")
print(df_stanford.nsmallest(10, "count")[["breed", "count"]].to_string(index=False))

### 1.3 Statistiques des images (taille, ratio, format)

In [ ]:
# Echantillonner des images pour les stats de taille (toutes serait trop long)
random.seed(42)
all_stanford_paths = [p for row in df_stanford["image_paths"] for p in row]
sample_stanford = random.sample(all_stanford_paths, min(3000, len(all_stanford_paths)))

widths, heights, ratios, modes = [], [], [], []
for p in sample_stanford:
    try:
        with Image.open(p) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            ratios.append(w / h)
            modes.append(img.mode)
    except Exception:
        continue

df_sizes = pd.DataFrame({"width": widths, "height": heights, "ratio": ratios, "mode": modes})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution des largeurs et hauteurs
axes[0].hist(df_sizes["width"], bins=50, alpha=0.7, label="Largeur", color="steelblue", edgecolor="white")
axes[0].hist(df_sizes["height"], bins=50, alpha=0.7, label="Hauteur", color="coral", edgecolor="white")
axes[0].set_xlabel("Pixels")
axes[0].set_ylabel("Frequence")
axes[0].set_title("Distribution des dimensions", fontweight="bold")
axes[0].legend()

# Scatter width vs height
axes[1].scatter(df_sizes["width"], df_sizes["height"], alpha=0.15, s=5, c="steelblue")
axes[1].set_xlabel("Largeur (px)")
axes[1].set_ylabel("Hauteur (px)")
axes[1].set_title("Largeur vs Hauteur", fontweight="bold")
axes[1].plot([0, df_sizes["width"].max()], [0, df_sizes["width"].max()], "r--", lw=1, label="ratio 1:1")
axes[1].legend()

# Distribution des ratios
axes[2].hist(df_sizes["ratio"], bins=50, color="mediumpurple", edgecolor="white", alpha=0.85)
axes[2].axvline(1.0, color="red", ls="--", lw=1.5, label="Carre (1:1)")
axes[2].set_xlabel("Ratio largeur/hauteur")
axes[2].set_ylabel("Frequence")
axes[2].set_title("Distribution des ratios d'aspect", fontweight="bold")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Dimensions (echantillon de {len(df_sizes)} images) :")
print(f"  Largeur  : {df_sizes['width'].mean():.0f} +/- {df_sizes['width'].std():.0f} px  "
      f"(min={df_sizes['width'].min()}, max={df_sizes['width'].max()})")
print(f"  Hauteur  : {df_sizes['height'].mean():.0f} +/- {df_sizes['height'].std():.0f} px  "
      f"(min={df_sizes['height'].min()}, max={df_sizes['height'].max()})")
print(f"  Ratio    : {df_sizes['ratio'].mean():.2f} +/- {df_sizes['ratio'].std():.2f}")
print(f"  Modes    : {Counter(df_sizes['mode'])}")

### 1.4 Distribution des reflectances (intensites RGB)

Analyse des valeurs de pixels sur un echantillon d'images pour comprendre la distribution des intensites par canal (R, G, B) et globale.

In [ ]:
# Calculer les stats de reflectance sur un echantillon
random.seed(42)
sample_refl = random.sample(all_stanford_paths, min(500, len(all_stanford_paths)))

channel_means = {"R": [], "G": [], "B": []}
channel_stds = {"R": [], "G": [], "B": []}
all_histograms = {"R": np.zeros(256), "G": np.zeros(256), "B": np.zeros(256)}

for p in sample_refl:
    try:
        with Image.open(p) as img:
            arr = np.array(img.convert("RGB"), dtype=np.float32)
            for i, ch in enumerate(["R", "G", "B"]):
                channel_means[ch].append(arr[:, :, i].mean())
                channel_stds[ch].append(arr[:, :, i].std())
                hist, _ = np.histogram(arr[:, :, i].ravel(), bins=256, range=(0, 256))
                all_histograms[ch] += hist
    except Exception:
        continue

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Histogramme global RGB
colors_rgb = {"R": "red", "G": "green", "B": "blue"}
for ch in ["R", "G", "B"]:
    h = all_histograms[ch]
    h_norm = h / h.sum()
    axes[0, 0].plot(range(256), h_norm, color=colors_rgb[ch], alpha=0.7, lw=1.5, label=ch)
axes[0, 0].set_xlabel("Valeur de pixel (0-255)")
axes[0, 0].set_ylabel("Densite")
axes[0, 0].set_title("Distribution globale des intensites RGB", fontweight="bold")
axes[0, 0].legend()

# Distribution des moyennes par image
for ch in ["R", "G", "B"]:
    axes[0, 1].hist(channel_means[ch], bins=40, alpha=0.5, color=colors_rgb[ch], label=ch, edgecolor="white")
axes[0, 1].set_xlabel("Moyenne d'intensite par image")
axes[0, 1].set_ylabel("Nombre d'images")
axes[0, 1].set_title("Distribution des moyennes par image (par canal)", fontweight="bold")
axes[0, 1].legend()

# Distribution des ecarts-types par image
for ch in ["R", "G", "B"]:
    axes[1, 0].hist(channel_stds[ch], bins=40, alpha=0.5, color=colors_rgb[ch], label=ch, edgecolor="white")
axes[1, 0].set_xlabel("Ecart-type d'intensite par image")
axes[1, 0].set_ylabel("Nombre d'images")
axes[1, 0].set_title("Distribution des ecarts-types par image", fontweight="bold")
axes[1, 0].legend()

# Boxplot des moyennes par canal
bp_data = [channel_means["R"], channel_means["G"], channel_means["B"]]
bp = axes[1, 1].boxplot(bp_data, labels=["R", "G", "B"], patch_artist=True,
                         medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], ["#ff6b6b", "#51cf66", "#339af0"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1, 1].set_ylabel("Moyenne d'intensite")
axes[1, 1].set_title("Boxplot des moyennes par canal", fontweight="bold")

plt.suptitle("Stanford Dogs — Reflectances", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Statistiques des reflectances (echantillon de {} images) :".format(len(sample_refl)))
for ch in ["R", "G", "B"]:
    m = np.array(channel_means[ch])
    s = np.array(channel_stds[ch])
    print(f"  {ch} : moyenne={m.mean():.1f} +/- {m.std():.1f},  "
          f"std intra-image={s.mean():.1f} +/- {s.std():.1f}")

---
## 2. Oxford-IIIT Pet Dataset (37 races)

### 2.1 Chargement et structure

In [ ]:
# Charger les annotations Oxford-IIIT Pet
# Format : IMAGE_NAME CLASS_ID SPECIES BREED_ID
# SPECIES: 1=Cat, 2=Dog
# Premiere lettre majuscule = chat, minuscule = chien

oxford_records = []
for split_file in ["trainval.txt", "test.txt"]:
    fpath = OXFORD_ANN_DIR / split_file
    split_name = "trainval" if "trainval" in split_file else "test"
    with open(fpath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            img_name = parts[0]
            class_id = int(parts[1])
            species = int(parts[2])  # 1=Cat, 2=Dog
            breed_id = int(parts[3])
            # Extraire le nom de race a partir du nom d'image (enlever le _NUM final)
            breed_name = "_".join(img_name.split("_")[:-1])
            oxford_records.append({
                "image_name": img_name,
                "class_id": class_id,
                "species": "Chat" if species == 1 else "Chien",
                "breed_id": breed_id,
                "breed": breed_name.replace("_", " "),
                "split": split_name,
            })

df_oxford = pd.DataFrame(oxford_records)

print(f"Total images      : {len(df_oxford)}")
print(f"  - trainval      : {(df_oxford['split'] == 'trainval').sum()}")
print(f"  - test          : {(df_oxford['split'] == 'test').sum()}")
print(f"Nombre de races   : {df_oxford['breed'].nunique()}")
print(f"Nombre de classes : {df_oxford['class_id'].nunique()}")
print(f"Especes           : {dict(df_oxford['species'].value_counts())}")
print(f"\nRaces par espece  :")
for sp in ["Chat", "Chien"]:
    breeds = df_oxford[df_oxford["species"] == sp]["breed"].unique()
    print(f"  {sp} : {len(breeds)} races")

### 2.2 Distribution des classes (especes et races)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Pie chart especes
species_counts = df_oxford["species"].value_counts()
axes[0].pie(species_counts, labels=species_counts.index, autopct="%1.1f%%",
            colors=["#ff9f43", "#54a0ff"], startangle=90, textprops={"fontsize": 12})
axes[0].set_title("Repartition Chats / Chiens", fontweight="bold")

# Bar chart par race
breed_counts = df_oxford.groupby(["breed", "species"]).size().reset_index(name="count")
breed_counts = breed_counts.sort_values("count", ascending=True)

colors_species = breed_counts["species"].map({"Chat": "#ff9f43", "Chien": "#54a0ff"})
axes[1].barh(range(len(breed_counts)), breed_counts["count"], color=colors_species, edgecolor="none")
axes[1].set_yticks(range(len(breed_counts)))
axes[1].set_yticklabels(breed_counts["breed"], fontsize=7)
axes[1].set_xlabel("Nombre d'images")
axes[1].set_title("Images par race (37 races)", fontweight="bold")
# Legende manuelle
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color="#ff9f43", label="Chat"), Patch(color="#54a0ff", label="Chien")],
               loc="lower right", fontsize=9)

# Histogramme de la distribution
breed_count_vals = breed_counts["count"]
axes[2].hist(breed_count_vals, bins=20, color="mediumpurple", edgecolor="white", alpha=0.85)
axes[2].axvline(breed_count_vals.mean(), color="red", ls="--", lw=2, label=f"Moyenne = {breed_count_vals.mean():.1f}")
axes[2].set_xlabel("Nombre d'images par race")
axes[2].set_ylabel("Nombre de races")
axes[2].set_title("Distribution du nombre d'images par race", fontweight="bold")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Images par race : min={breed_count_vals.min()}, max={breed_count_vals.max()}, "
      f"mean={breed_count_vals.mean():.1f}, std={breed_count_vals.std():.1f}")
print(f"Ratio desequilibre : {breed_count_vals.max() / breed_count_vals.min():.2f}")

### 2.3 Distribution des masques trimap (segmentation)

In [ ]:
# Analyser la distribution des pixels dans les trimaps
# Trimap : 1 = foreground (animal), 2 = background, 3 = border
random.seed(42)
trimap_files = sorted(OXFORD_TRIMAP_DIR.glob("*.png"))
sample_trimaps = random.sample(trimap_files, min(500, len(trimap_files)))

class_pixel_counts = {1: 0, 2: 0, 3: 0}
class_ratios = {1: [], 2: [], 3: []}
class_names_trimap = {1: "Foreground (animal)", 2: "Background", 3: "Border"}

for tp in sample_trimaps:
    try:
        arr = np.array(Image.open(tp))
        total_px = arr.size
        for cls in [1, 2, 3]:
            cnt = (arr == cls).sum()
            class_pixel_counts[cls] += cnt
            class_ratios[cls].append(cnt / total_px * 100)
    except Exception:
        continue

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pie chart des pixels par classe
labels = [class_names_trimap[k] for k in [1, 2, 3]]
sizes = [class_pixel_counts[k] for k in [1, 2, 3]]
colors_trimap = ["#2ecc71", "#e74c3c", "#f39c12"]
axes[0].pie(sizes, labels=labels, autopct="%1.1f%%", colors=colors_trimap,
            startangle=90, textprops={"fontsize": 11})
axes[0].set_title("Repartition des pixels (trimap)", fontweight="bold")

# Distribution du ratio foreground par image
axes[1].hist(class_ratios[1], bins=40, color="#2ecc71", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("% de pixels foreground par image")
axes[1].set_ylabel("Nombre d'images")
axes[1].set_title("Distribution du ratio foreground", fontweight="bold")
axes[1].axvline(np.mean(class_ratios[1]), color="red", ls="--", lw=1.5,
                label=f"Moyenne = {np.mean(class_ratios[1]):.1f}%")
axes[1].legend()

# Boxplot des 3 classes
bp = axes[2].boxplot([class_ratios[1], class_ratios[2], class_ratios[3]],
                      labels=["Foreground", "Background", "Border"],
                      patch_artist=True, medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], colors_trimap):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[2].set_ylabel("% de pixels par image")
axes[2].set_title("Distribution des classes trimap par image", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Trimaps analyses : {len(sample_trimaps)}")
for cls in [1, 2, 3]:
    r = np.array(class_ratios[cls])
    print(f"  {class_names_trimap[cls]:25s} : {r.mean():.1f}% +/- {r.std():.1f}%")

### 2.4 Statistiques des images Oxford (taille, ratio)

In [ ]:
# Stats de taille sur les images Oxford
random.seed(42)
oxford_img_files = sorted(OXFORD_IMG_DIR.glob("*.jpg"))
sample_oxford_imgs = random.sample(oxford_img_files, min(2000, len(oxford_img_files)))

ox_widths, ox_heights, ox_ratios, ox_modes = [], [], [], []
for p in sample_oxford_imgs:
    try:
        with Image.open(p) as img:
            w, h = img.size
            ox_widths.append(w)
            ox_heights.append(h)
            ox_ratios.append(w / h)
            ox_modes.append(img.mode)
    except Exception:
        continue

df_ox_sizes = pd.DataFrame({"width": ox_widths, "height": ox_heights, "ratio": ox_ratios, "mode": ox_modes})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_ox_sizes["width"], bins=50, alpha=0.7, label="Largeur", color="steelblue", edgecolor="white")
axes[0].hist(df_ox_sizes["height"], bins=50, alpha=0.7, label="Hauteur", color="coral", edgecolor="white")
axes[0].set_xlabel("Pixels")
axes[0].set_ylabel("Frequence")
axes[0].set_title("Distribution des dimensions (Oxford)", fontweight="bold")
axes[0].legend()

axes[1].scatter(df_ox_sizes["width"], df_ox_sizes["height"], alpha=0.15, s=5, c="steelblue")
axes[1].set_xlabel("Largeur (px)")
axes[1].set_ylabel("Hauteur (px)")
axes[1].set_title("Largeur vs Hauteur", fontweight="bold")
axes[1].plot([0, df_ox_sizes["width"].max()], [0, df_ox_sizes["width"].max()], "r--", lw=1, label="ratio 1:1")
axes[1].legend()

axes[2].hist(df_ox_sizes["ratio"], bins=50, color="mediumpurple", edgecolor="white", alpha=0.85)
axes[2].axvline(1.0, color="red", ls="--", lw=1.5, label="Carre (1:1)")
axes[2].set_xlabel("Ratio largeur/hauteur")
axes[2].set_ylabel("Frequence")
axes[2].set_title("Distribution des ratios d'aspect", fontweight="bold")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Dimensions (echantillon de {len(df_ox_sizes)} images) :")
print(f"  Largeur  : {df_ox_sizes['width'].mean():.0f} +/- {df_ox_sizes['width'].std():.0f} px  "
      f"(min={df_ox_sizes['width'].min()}, max={df_ox_sizes['width'].max()})")
print(f"  Hauteur  : {df_ox_sizes['height'].mean():.0f} +/- {df_ox_sizes['height'].std():.0f} px  "
      f"(min={df_ox_sizes['height'].min()}, max={df_ox_sizes['height'].max()})")
print(f"  Ratio    : {df_ox_sizes['ratio'].mean():.2f} +/- {df_ox_sizes['ratio'].std():.2f}")
print(f"  Modes    : {Counter(df_ox_sizes['mode'])}")

### 2.5 Distribution des reflectances (intensites RGB) — Oxford

In [ ]:
# Reflectances Oxford
random.seed(42)
sample_ox_refl = random.sample(oxford_img_files, min(500, len(oxford_img_files)))

ox_ch_means = {"R": [], "G": [], "B": []}
ox_ch_stds = {"R": [], "G": [], "B": []}
ox_histograms = {"R": np.zeros(256), "G": np.zeros(256), "B": np.zeros(256)}

for p in sample_ox_refl:
    try:
        with Image.open(p) as img:
            arr = np.array(img.convert("RGB"), dtype=np.float32)
            for i, ch in enumerate(["R", "G", "B"]):
                ox_ch_means[ch].append(arr[:, :, i].mean())
                ox_ch_stds[ch].append(arr[:, :, i].std())
                hist, _ = np.histogram(arr[:, :, i].ravel(), bins=256, range=(0, 256))
                ox_histograms[ch] += hist
    except Exception:
        continue

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ch in ["R", "G", "B"]:
    h = ox_histograms[ch]
    h_norm = h / h.sum()
    axes[0, 0].plot(range(256), h_norm, color=colors_rgb[ch], alpha=0.7, lw=1.5, label=ch)
axes[0, 0].set_xlabel("Valeur de pixel (0-255)")
axes[0, 0].set_ylabel("Densite")
axes[0, 0].set_title("Distribution globale des intensites RGB", fontweight="bold")
axes[0, 0].legend()

for ch in ["R", "G", "B"]:
    axes[0, 1].hist(ox_ch_means[ch], bins=40, alpha=0.5, color=colors_rgb[ch], label=ch, edgecolor="white")
axes[0, 1].set_xlabel("Moyenne d'intensite par image")
axes[0, 1].set_ylabel("Nombre d'images")
axes[0, 1].set_title("Distribution des moyennes par image (par canal)", fontweight="bold")
axes[0, 1].legend()

for ch in ["R", "G", "B"]:
    axes[1, 0].hist(ox_ch_stds[ch], bins=40, alpha=0.5, color=colors_rgb[ch], label=ch, edgecolor="white")
axes[1, 0].set_xlabel("Ecart-type d'intensite par image")
axes[1, 0].set_ylabel("Nombre d'images")
axes[1, 0].set_title("Distribution des ecarts-types par image", fontweight="bold")
axes[1, 0].legend()

bp_data = [ox_ch_means["R"], ox_ch_means["G"], ox_ch_means["B"]]
bp = axes[1, 1].boxplot(bp_data, labels=["R", "G", "B"], patch_artist=True,
                         medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], ["#ff6b6b", "#51cf66", "#339af0"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1, 1].set_ylabel("Moyenne d'intensite")
axes[1, 1].set_title("Boxplot des moyennes par canal", fontweight="bold")

plt.suptitle("Oxford-IIIT Pet — Reflectances", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Statistiques des reflectances (echantillon de {} images) :".format(len(sample_ox_refl)))
for ch in ["R", "G", "B"]:
    m = np.array(ox_ch_means[ch])
    s = np.array(ox_ch_stds[ch])
    print(f"  {ch} : moyenne={m.mean():.1f} +/- {m.std():.1f},  "
          f"std intra-image={s.mean():.1f} +/- {s.std():.1f}")

---
## 3. Comparaison inter-datasets

### 3.1 Tableau recapitulatif

In [ ]:
# Tableau recapitulatif
summary = pd.DataFrame({
    "Metrique": [
        "Total images",
        "Nombre de classes",
        "Min images / classe",
        "Max images / classe",
        "Moyenne images / classe",
        "Ecart-type images / classe",
        "Ratio desequilibre (max/min)",
        "Largeur moyenne (px)",
        "Hauteur moyenne (px)",
        "Ratio d'aspect moyen",
        "Moyenne R",
        "Moyenne G",
        "Moyenne B",
    ],
    "Stanford Dogs": [
        total_stanford,
        len(df_stanford),
        df_stanford["count"].min(),
        df_stanford["count"].max(),
        f"{df_stanford['count'].mean():.1f}",
        f"{df_stanford['count'].std():.1f}",
        f"{df_stanford['count'].max() / df_stanford['count'].min():.2f}",
        f"{df_sizes['width'].mean():.0f}",
        f"{df_sizes['height'].mean():.0f}",
        f"{df_sizes['ratio'].mean():.2f}",
        f"{np.mean(channel_means['R']):.1f}",
        f"{np.mean(channel_means['G']):.1f}",
        f"{np.mean(channel_means['B']):.1f}",
    ],
    "Oxford-IIIT Pet": [
        len(df_oxford),
        df_oxford["breed"].nunique(),
        breed_count_vals.min(),
        breed_count_vals.max(),
        f"{breed_count_vals.mean():.1f}",
        f"{breed_count_vals.std():.1f}",
        f"{breed_count_vals.max() / breed_count_vals.min():.2f}",
        f"{df_ox_sizes['width'].mean():.0f}",
        f"{df_ox_sizes['height'].mean():.0f}",
        f"{df_ox_sizes['ratio'].mean():.2f}",
        f"{np.mean(ox_ch_means['R']):.1f}",
        f"{np.mean(ox_ch_means['G']):.1f}",
        f"{np.mean(ox_ch_means['B']):.1f}",
    ],
})

summary.style.set_properties(**{"text-align": "center"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "center")]}]
)

### 3.2 Comparaison des distributions de taille et d'intensite

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# --- Ligne 1 : Comparaison des tailles ---
# Largeurs
axes[0, 0].hist(df_sizes["width"], bins=50, alpha=0.6, label="Stanford", color="steelblue", edgecolor="white", density=True)
axes[0, 0].hist(df_ox_sizes["width"], bins=50, alpha=0.6, label="Oxford", color="coral", edgecolor="white", density=True)
axes[0, 0].set_xlabel("Largeur (px)")
axes[0, 0].set_ylabel("Densite")
axes[0, 0].set_title("Distribution des largeurs", fontweight="bold")
axes[0, 0].legend()

# Hauteurs
axes[0, 1].hist(df_sizes["height"], bins=50, alpha=0.6, label="Stanford", color="steelblue", edgecolor="white", density=True)
axes[0, 1].hist(df_ox_sizes["height"], bins=50, alpha=0.6, label="Oxford", color="coral", edgecolor="white", density=True)
axes[0, 1].set_xlabel("Hauteur (px)")
axes[0, 1].set_ylabel("Densite")
axes[0, 1].set_title("Distribution des hauteurs", fontweight="bold")
axes[0, 1].legend()

# Ratios
axes[0, 2].hist(df_sizes["ratio"], bins=50, alpha=0.6, label="Stanford", color="steelblue", edgecolor="white", density=True)
axes[0, 2].hist(df_ox_sizes["ratio"], bins=50, alpha=0.6, label="Oxford", color="coral", edgecolor="white", density=True)
axes[0, 2].axvline(1.0, color="black", ls="--", lw=1, alpha=0.5)
axes[0, 2].set_xlabel("Ratio largeur/hauteur")
axes[0, 2].set_ylabel("Densite")
axes[0, 2].set_title("Distribution des ratios d'aspect", fontweight="bold")
axes[0, 2].legend()

# --- Ligne 2 : Comparaison des reflectances par canal ---
for idx, ch in enumerate(["R", "G", "B"]):
    # Stanford
    h_s = all_histograms[ch]
    h_s_norm = h_s / h_s.sum()
    axes[1, idx].plot(range(256), h_s_norm, color=colors_rgb[ch], alpha=0.8, lw=1.5, label=f"Stanford", ls="-")
    # Oxford
    h_o = ox_histograms[ch]
    h_o_norm = h_o / h_o.sum()
    axes[1, idx].plot(range(256), h_o_norm, color=colors_rgb[ch], alpha=0.8, lw=1.5, label=f"Oxford", ls="--")
    axes[1, idx].set_xlabel("Valeur de pixel")
    axes[1, idx].set_ylabel("Densite")
    axes[1, idx].set_title(f"Canal {ch} — Stanford vs Oxford", fontweight="bold")
    axes[1, idx].legend()
    axes[1, idx].fill_between(range(256), h_s_norm, alpha=0.15, color=colors_rgb[ch])
    axes[1, idx].fill_between(range(256), h_o_norm, alpha=0.1, color=colors_rgb[ch])

plt.suptitle("Comparaison Stanford Dogs vs Oxford-IIIT Pet", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Test statistique de Kolmogorov-Smirnov sur les distributions de reflectance

In [ ]:
# Test KS pour comparer les distributions de reflectance entre les deux datasets
print("Test de Kolmogorov-Smirnov (H0 : les deux distributions sont identiques)")
print("=" * 70)
for ch in ["R", "G", "B"]:
    ks_stat, p_value = sp_stats.ks_2samp(channel_means[ch], ox_ch_means[ch])
    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"  Canal {ch} : KS={ks_stat:.4f}, p={p_value:.2e}  {sig}")

print("\n" + "=" * 70)
print("Test KS sur les distributions de taille")
for dim, s_data, o_data in [("Largeur", df_sizes["width"], df_ox_sizes["width"]),
                              ("Hauteur", df_sizes["height"], df_ox_sizes["height"]),
                              ("Ratio",   df_sizes["ratio"],  df_ox_sizes["ratio"])]:
    ks_stat, p_value = sp_stats.ks_2samp(s_data, o_data)
    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"  {dim:8s} : KS={ks_stat:.4f}, p={p_value:.2e}  {sig}")

print("\nLegende : *** p<0.001, ** p<0.01, * p<0.05, ns = non significatif")

### 3.4 Exemples d'images des trois datasets

In [ ]:
random.seed(42)

fig, axes = plt.subplots(3, 6, figsize=(20, 10))

# Stanford Dogs - 6 images aleatoires
sample_stanford_show = random.sample(all_stanford_paths, 6)
for i, p in enumerate(sample_stanford_show):
    img = Image.open(p).convert("RGB")
    breed = p.parent.name.split("-", 1)[1].replace("_", " ")
    axes[0, i].imshow(img)
    axes[0, i].set_title(breed, fontsize=8)
    axes[0, i].axis("off")
axes[0, 0].set_ylabel("Stanford Dogs", fontsize=12, fontweight="bold", rotation=0, labelpad=100)

# Oxford Pet - 6 images aleatoires
sample_oxford_show = random.sample(list(OXFORD_IMG_DIR.glob("*.jpg")), 6)
for i, p in enumerate(sample_oxford_show):
    img = Image.open(p).convert("RGB")
    breed = "_".join(p.stem.split("_")[:-1]).replace("_", " ")
    axes[1, i].imshow(img)
    axes[1, i].set_title(breed, fontsize=8)
    axes[1, i].axis("off")
axes[1, 0].set_ylabel("Oxford Pet", fontsize=12, fontweight="bold", rotation=0, labelpad=100)

# Oxford Trimaps correspondants
for i, p in enumerate(sample_oxford_show):
    trimap_path = OXFORD_TRIMAP_DIR / (p.stem + ".png")
    if trimap_path.exists():
        trimap = np.array(Image.open(trimap_path))
        # Coloriser le trimap : 1=vert (foreground), 2=rouge (bg), 3=jaune (border)
        colored = np.zeros((*trimap.shape, 3), dtype=np.uint8)
        colored[trimap == 1] = [46, 204, 113]   # vert
        colored[trimap == 2] = [231, 76, 60]     # rouge
        colored[trimap == 3] = [243, 156, 18]    # jaune
        axes[2, i].imshow(colored)
    axes[2, i].axis("off")
axes[2, 0].set_ylabel("Trimaps", fontsize=12, fontweight="bold", rotation=0, labelpad=100)

plt.suptitle("Exemples d'images par dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Recapitulatif et points cles

### Vue d'ensemble

In [ ]:
# ============================================================
# RECAPITULATIF COMPLET
# ============================================================

from IPython.display import display, Markdown

# --- Stanford Dogs ---
st_imbalance = df_stanford["count"].max() / df_stanford["count"].min()
st_mean_r = np.mean(channel_means["R"])
st_mean_g = np.mean(channel_means["G"])
st_mean_b = np.mean(channel_means["B"])

# --- Oxford Pet ---
ox_imbalance = breed_count_vals.max() / breed_count_vals.min()
n_cats = (df_oxford["species"] == "Chat").sum()
n_dogs = (df_oxford["species"] == "Chien").sum()
pct_cats = 100 * n_cats / len(df_oxford)
pct_dogs = 100 * n_dogs / len(df_oxford)
ox_mean_r = np.mean(ox_ch_means["R"])
ox_mean_g = np.mean(ox_ch_means["G"])
ox_mean_b = np.mean(ox_ch_means["B"])
fg_mean = np.mean(class_ratios[1])
bg_mean = np.mean(class_ratios[2])
border_mean = np.mean(class_ratios[3])

recap = f"""
# Recapitulatif — Statistiques des datasets

---

## Stanford Dogs

| Caracteristique | Valeur |
|---|---|
| **Nombre d'images** | {total_stanford} |
| **Nombre de races** | {len(df_stanford)} |
| **Images / race** | {df_stanford['count'].min()} – {df_stanford['count'].max()} (moy. {df_stanford['count'].mean():.0f} ± {df_stanford['count'].std():.0f}) |
| **Ratio de desequilibre** | {st_imbalance:.2f}x |
| **Dimensions moyennes** | {df_sizes['width'].mean():.0f} x {df_sizes['height'].mean():.0f} px |
| **Ratio d'aspect moyen** | {df_sizes['ratio'].mean():.2f} |
| **Reflectance moyenne (R/G/B)** | {st_mean_r:.1f} / {st_mean_g:.1f} / {st_mean_b:.1f} |

**Points cles :**
- Dataset **relativement equilibre** (ratio max/min ~ {st_imbalance:.1f}x) — pas besoin de sur-echantillonnage agressif
- Les images ont des **tailles tres variables** → le redimensionnement (224 ou 256 px) est indispensable
- Les reflectances montrent un **leger biais vers les tons chauds** (R > G > B), typique de photos d'animaux en interieur/exterieur

---

## Oxford-IIIT Pet

| Caracteristique | Valeur |
|---|---|
| **Nombre d'images** | {len(df_oxford)} |
| **Nombre de races** | {df_oxford['breed'].nunique()} (dont {df_oxford[df_oxford['species']=='Chat']['breed'].nunique()} chats, {df_oxford[df_oxford['species']=='Chien']['breed'].nunique()} chiens) |
| **Repartition especes** | {pct_cats:.1f}% chats / {pct_dogs:.1f}% chiens |
| **Images / race** | {breed_count_vals.min()} – {breed_count_vals.max()} (moy. {breed_count_vals.mean():.0f} ± {breed_count_vals.std():.0f}) |
| **Ratio de desequilibre** | {ox_imbalance:.2f}x |
| **Dimensions moyennes** | {df_ox_sizes['width'].mean():.0f} x {df_ox_sizes['height'].mean():.0f} px |
| **Ratio d'aspect moyen** | {df_ox_sizes['ratio'].mean():.2f} |
| **Reflectance moyenne (R/G/B)** | {ox_mean_r:.1f} / {ox_mean_g:.1f} / {ox_mean_b:.1f} |
| **Trimap — foreground** | {fg_mean:.1f}% des pixels |
| **Trimap — background** | {bg_mean:.1f}% des pixels |
| **Trimap — border** | {border_mean:.1f}% des pixels |

**Points cles :**
- Dataset **tres equilibre** (ratio ~ {ox_imbalance:.1f}x) avec ~200 images par race
- Contient **chats et chiens** — pour le pipeline BCS, seuls les chiens sont utiles (filtrage par `species == 2`)
- Les trimaps ont un **border tres fin** (~{border_mean:.1f}% des pixels) → la classe border est tres minoritaire, ce qui explique qu'elle est la plus difficile a segmenter
- Le **foreground** represente en moyenne ~{fg_mean:.0f}% de l'image — les animaux occupent une part significative du champ

---

## Comparaison et implications pour le pipeline

| Aspect | Stanford Dogs | Oxford Pet | Impact pipeline |
|---|---|---|---|
| **Tache** | Classification (race) | Segmentation (trimap) | Deux branches distinctes du pipeline |
| **Taille** | ~20k images | ~7k images | Stanford permet un entrainement plus robuste |
| **Equilibre** | Bon (ratio ~{st_imbalance:.1f}x) | Tres bon (ratio ~{ox_imbalance:.1f}x) | Pas de class weighting necessaire |
| **Diversite tailles** | Haute ({df_sizes['width'].std():.0f} px std) | Haute ({df_ox_sizes['width'].std():.0f} px std) | Resize + crop obligatoire |
| **Profil colorimetrique** | R={st_mean_r:.0f}/G={st_mean_g:.0f}/B={st_mean_b:.0f} | R={ox_mean_r:.0f}/G={ox_mean_g:.0f}/B={ox_mean_b:.0f} | Normalisation ImageNet applicable aux deux |
| **Nb classes** | 120 races | 37 races (3 seg) | La classification de race est le goulot (120 classes, 170 img/classe) |

**Recommandations :**
1. **Normalisation** : les moyennes RGB des deux datasets sont proches des stats ImageNet — la normalisation standard (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) est adaptee
2. **Augmentation** : la variabilite des tailles et ratios justifie l'utilisation de RandomResizedCrop + flips horizontaux
3. **Segmentation** : le desequilibre foreground/background/border justifie l'utilisation d'une loss combinee (CE + Dice) deja en place dans `LitSegmentationModule`
4. **Classification** : avec ~170 images/classe sur 120 classes, le transfer learning (ViT ou ResNet pretrained) est essentiel — un entrainement from scratch serait sous-determine
"""

display(Markdown(recap))